# Climatological Stations Analysis - EH 26289

This notebook processes daily precipitation data from 16 climatological stations in the Río Tempoal and surrounding basins. The analysis includes:
1. Downloading daily precipitation data from CONAGUA URLs
2. Filtering stations with data extending to 2015 and at least 20 years of records
3. Creating monthly precipitation DataFrames with year index and month columns

## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import requests
from pathlib import Path
from datetime import datetime
from urllib.parse import urlparse
import urllib3

# Disable SSL verification warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

## 2. Load and Explore Estaciones CSV

In [2]:
# Read the estaciones.csv file
estaciones = pd.read_csv('../data/precipitacion/estaciones.csv')

# Check the DIARIOS column
print("DIARIOS column sample:")
print(estaciones[['CLAVE', 'NOMBRE', 'DIARIOS']].head(10))
print(f"\nTotal rows: {len(estaciones)}")
print(f"Non-null DIARIOS entries: {estaciones['DIARIOS'].notna().sum()}")

DIARIOS column sample:
   CLAVE                       NOMBRE  \
0  13011                     HUEJUTLA   
1  13015   SAN AGUSTIN METZIQUITITLAN   
2  13042            ZACUALTIPAN (SMN)   
3  13077                   METZTITLAN   
4  13087                SAN CRISTOBAL   
5  13093                      VENADOS   
6  13135                    ATLAPEXCO   
7  13139                HUAUTLA (DGE)   
8  30016                BENITO JUAREZ   
9  30041  CHICONTEPEC DE TEJEDA (SMN)   

                                             DIARIOS  
0  https://smn.conagua.gob.mx/tools/RESOURCES/Nor...  
1  https://smn.conagua.gob.mx/tools/RESOURCES/Nor...  
2  https://smn.conagua.gob.mx/tools/RESOURCES/Nor...  
3  https://smn.conagua.gob.mx/tools/RESOURCES/Nor...  
4  https://smn.conagua.gob.mx/tools/RESOURCES/Nor...  
5  https://smn.conagua.gob.mx/tools/RESOURCES/Nor...  
6  https://smn.conagua.gob.mx/tools/RESOURCES/Nor...  
7  https://smn.conagua.gob.mx/tools/RESOURCES/Nor...  
8  https://smn.conagua.gob.mx/

## 3. Download Daily Precipitation Files from DIARIOS URLs

In [3]:
'''# Read the estaciones.csv file
estaciones = pd.read_csv('data/precipitacion/estaciones.csv')

# Target folder
target_folder = Path('data/precipitacion')
target_folder.mkdir(parents=True, exist_ok=True)

# Download each file
download_count = 0
error_count = 0

for idx, row in estaciones.iterrows():
    url = row['DIARIOS']
    clave = row['CLAVE']
    
    if pd.isna(url):
        print(f"Skipping row {idx}: No URL for {clave}")
        continue
    
    try:
        # Extract filename from URL
        filename = urlparse(url).path.split('/')[-1]
        filepath = target_folder / filename
        
        print(f"Downloading {clave}: {filename}...", end=' ')
        
        # Download the file with SSL verification disabled
        response = requests.get(url, timeout=10, verify=False)
        response.raise_for_status()
        
        # Save the file (overwriting if it exists)
        with open(filepath, 'wb') as f:
            f.write(response.content)
        
        print(f"✓ ({len(response.content)} bytes)")
        download_count += 1
        
    except Exception as e:
        print(f"✗ Error: {str(e)}")
        error_count += 1

print(f"\n--- Summary ---")
print(f"Successfully downloaded: {download_count}")
print(f"Errors: {error_count}")
print(f"Total: {len(estaciones)}")'''

'# Read the estaciones.csv file\nestaciones = pd.read_csv(\'data/precipitacion/estaciones.csv\')\n\n# Target folder\ntarget_folder = Path(\'data/precipitacion\')\ntarget_folder.mkdir(parents=True, exist_ok=True)\n\n# Download each file\ndownload_count = 0\nerror_count = 0\n\nfor idx, row in estaciones.iterrows():\n    url = row[\'DIARIOS\']\n    clave = row[\'CLAVE\']\n\n    if pd.isna(url):\n        print(f"Skipping row {idx}: No URL for {clave}")\n        continue\n\n    try:\n        # Extract filename from URL\n        filename = urlparse(url).path.split(\'/\')[-1]\n        filepath = target_folder / filename\n\n        print(f"Downloading {clave}: {filename}...", end=\' \')\n\n        # Download the file with SSL verification disabled\n        response = requests.get(url, timeout=10, verify=False)\n        response.raise_for_status()\n\n        # Save the file (overwriting if it exists)\n        with open(filepath, \'wb\') as f:\n            f.write(response.content)\n\n        pr

## 4. Analyze Climatological Stations with Data Extending to 2015

In [4]:
# Read the estaciones.csv
estaciones = pd.read_csv('../data/precipitacion/estaciones.csv')

# Filter for climatological stations only
climatologicas = estaciones[estaciones['TIPO_EST'] == 'CLIMATOLÓGICA'].copy()

print(f"Total climatological stations: {len(climatologicas)}")
print(f"\nAnalyzing daily data files for year range...\n")

# Check each file
results = []
data_folder = Path('../data/precipitacion')

for idx, row in climatologicas.iterrows():
    clave = row['CLAVE']
    nombre = row['NOMBRE']
    filename = f"dia{clave}.txt"
    filepath = data_folder / filename
    
    if not filepath.exists():
        print(f"❌ {clave}: File not found")
        continue
    
    try:
        # Read the file using a byte-safe Latin-1 decode to avoid Windows charmap errors
        with open(filepath, 'r', encoding='latin-1') as f:
            lines = f.readlines()
        
        # Find lines that look like dates
        dates = []
        for line in lines:
            parts = line.split()
            if len(parts) > 0:
                date_str = parts[0]
                try:
                    # Try parsing as YYYY/MM/DD
                    if '/' in date_str:
                        date_obj = datetime.strptime(date_str, '%Y/%m/%d')
                        dates.append(date_obj)
                    elif '-' in date_str and len(date_str) == 10:
                        date_obj = datetime.strptime(date_str, '%Y-%m-%d')
                        dates.append(date_obj)
                except:
                    pass
        
        if dates:
            min_year = min(dates).year
            max_year = max(dates).year
            
            has_2015 = any(d.year >= 2015 for d in dates)
            
            status = "✓" if has_2015 else "✗"
            print(f"{status} {clave}: {nombre:40s} | {min_year} - {max_year}")
            
            if has_2015:
                results.append({
                    'CLAVE': clave,
                    'NOMBRE': nombre,
                    'MIN_YEAR': min_year,
                    'MAX_YEAR': max_year
                })
        else:
            print(f"❓ {clave}: {nombre:40s} | Could not parse dates")
    
    except Exception as e:
        print(f"❌ {clave}: Error - {str(e)[:50]}")

print(f"\n{'='*70}")
print(f"Climatological stations with data including 2015 or later:")
print(f"{'='*70}")
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
print(f"\nTotal selected: {len(results_df)}")

Total climatological stations: 28

Analyzing daily data files for year range...

✓ 13011: HUEJUTLA                                 | 1969 - 2026
✓ 13015: SAN AGUSTIN METZIQUITITLAN               | 1942 - 2025
✓ 13042: ZACUALTIPAN (SMN)                        | 1941 - 2026
✓ 13077: METZTITLAN                               | 1952 - 2026
✓ 13087: SAN CRISTOBAL                            | 1954 - 2025
✓ 13093: VENADOS                                  | 1951 - 2026
✓ 13135: ATLAPEXCO                                | 1981 - 2026
✓ 13139: HUAUTLA (DGE)                            | 1981 - 2025
✓ 30016: BENITO JUAREZ                            | 1962 - 2022
✓ 30041: CHICONTEPEC DE TEJEDA (SMN)              | 1947 - 2023
✓ 30067: HUAYACOCOTLA                             | 1961 - 2026
✓ 30098: LOS HULES                                | 1960 - 2026
✓ 30180: TERRERILLOS                              | 1961 - 2026
✓ 30382: ZONTECOMATLAN                            | 1983 - 2019
✗ 13003: CALNALI       

## 5. Select Stations with at Least 20 Years of Records

In [5]:
# Read the estaciones.csv
estaciones = pd.read_csv('../data/precipitacion/estaciones.csv')

# Filter for climatological stations only
climatologicas = estaciones[estaciones['TIPO_EST'] == 'CLIMATOLÓGICA'].copy()

print(f"Total climatological stations: {len(climatologicas)}")
print(f"\nAnalyzing unique years in each station's data...\n")

# Check each file
results = []
data_folder = Path('../data/precipitacion')

for idx, row in climatologicas.iterrows():
    clave = row['CLAVE']
    nombre = row['NOMBRE']
    filename = f"dia{clave}.txt"
    filepath = data_folder / filename

    if not filepath.exists():
        print(f"❌ {clave}: File not found")
        continue

    try:
        # Read the file using a byte-safe Latin-1 decode to avoid Windows charmap errors
        with open(filepath, 'r', encoding='latin-1') as f:
            lines = f.readlines()

        # Extract unique years from all data lines
        unique_years = set()
        for line in lines:
            parts = line.split()
            if len(parts) > 0:
                date_str = parts[0]
                try:
                    # Try parsing as YYYY/MM/DD
                    if '/' in date_str:
                        date_obj = datetime.strptime(date_str, '%Y/%m/%d')
                        unique_years.add(date_obj.year)
                    elif '-' in date_str and len(date_str) == 10:
                        date_obj = datetime.strptime(date_str, '%Y-%m-%d')
                        unique_years.add(date_obj.year)
                except:
                    pass

        if unique_years:
            num_years = len(unique_years)
            min_year = min(unique_years)
            max_year = max(unique_years)
            span = max_year - min_year + 1

            has_20_years = num_years >= 20

            status = "✓" if has_20_years else "✗"
            print(f"{status} {clave}: {nombre:40s} | {num_years:2d} unique years ({min_year}-{max_year}, span: {span} years)")

            if has_20_years:
                results.append({
                    'CLAVE': clave,
                    'NOMBRE': nombre,
                    'UNIQUE_YEARS': num_years,
                    'MIN_YEAR': min_year,
                    'MAX_YEAR': max_year,
                    'SPAN': span
                })
        else:
            print(f"❓ {clave}: {nombre:40s} | Could not parse dates")

    except Exception as e:
        print(f"❌ {clave}: Error - {str(e)[:50]}")

print(f"\n{'='*80}")
print("Climatological stations with at least 20 years of records (continuous or discontinuous):")
print(f"{'='*80}")
results_df = pd.DataFrame(results)
if len(results_df) > 0:
    results_df = results_df.sort_values('UNIQUE_YEARS', ascending=False)
    print(results_df.to_string(index=False))
    print(f"\nTotal selected: {len(results_df)}")
else:
    print("No stations with 20+ years of records found.")

Total climatological stations: 28

Analyzing unique years in each station's data...

✓ 13011: HUEJUTLA                                 | 58 unique years (1969-2026, span: 58 years)
✓ 13015: SAN AGUSTIN METZIQUITITLAN               | 53 unique years (1942-2025, span: 84 years)
✓ 13042: ZACUALTIPAN (SMN)                        | 86 unique years (1941-2026, span: 86 years)
✓ 13077: METZTITLAN                               | 69 unique years (1952-2026, span: 75 years)
✓ 13087: SAN CRISTOBAL                            | 72 unique years (1954-2025, span: 72 years)
✓ 13093: VENADOS                                  | 72 unique years (1951-2026, span: 76 years)
✓ 13135: ATLAPEXCO                                | 44 unique years (1981-2026, span: 46 years)
✓ 13139: HUAUTLA (DGE)                            | 45 unique years (1981-2025, span: 45 years)
✓ 30016: BENITO JUAREZ                            | 53 unique years (1962-2022, span: 61 years)
✓ 30041: CHICONTEPEC DE TEJEDA (SMN)              |

## 6. Final Selection: Stations with 2015+ Data AND 20+ Years of Records

In [6]:
# The 16 selected stations meeting both criteria:
# - Data extending to at least 2015
# - At least 20 years of records (continuous or discontinuous)

selected_stations = [
    '13042', '30041', '13087', '13093', '13077', '30180', 
    '30098', '13011', '13015', '30016', '30067', '13139', 
    '13135', '30359', '13137', '30382'
]

print(f"Selected {len(selected_stations)} climatological stations:")
print(selected_stations)

Selected 16 climatological stations:
['13042', '30041', '13087', '13093', '13077', '30180', '30098', '13011', '13015', '30016', '30067', '13139', '13135', '30359', '13137', '30382']


## 7. Create Monthly Precipitation DataFrames

In [7]:
# List of the 16 selected stations
selected_stations = [
    '13042', '30041', '13087', '13093', '13077', '30180', 
    '30098', '13011', '13015', '30016', '30067', '13139', 
    '13135', '30359', '13137', '30382'
]

data_folder = Path('../data/precipitacion')
dfs_monthly = {}

print(f"Processing {len(selected_stations)} stations...\n")

for clave in selected_stations:
    filename = f"dia{clave}.txt"
    filepath = data_folder / filename
    
    if not filepath.exists():
        print(f"❌ {clave}: File not found")
        continue
    
    try:
        # Read the file using a byte-safe Latin-1 decode to avoid Windows charmap errors
        with open(filepath, 'r', encoding='latin-1') as f:
            lines = f.readlines()
        
        # Parse the data - extract dates and precipitation values
        dates = []
        precip = []
        
        for line in lines:
            parts = line.split()
            if len(parts) >= 2:
                date_str = parts[0]
                try:
                    # Try parsing as YYYY/MM/DD
                    if '/' in date_str:
                        date_obj = datetime.strptime(date_str, '%Y/%m/%d')
                    elif '-' in date_str and len(date_str) == 10:
                        date_obj = datetime.strptime(date_str, '%Y-%m-%d')
                    else:
                        continue
                    
                    # Read precipitation only from the precipitation column.
                    precip_token = parts[1]
                    if precip_token == 'NULO':
                        precip_val = np.nan
                    else:
                        try:
                            precip_val = float(precip_token)
                        except:
                            precip_val = np.nan
                    
                    if not pd.isna(precip_val):
                        dates.append(date_obj)
                        precip.append(precip_val)
                except:
                    pass
        
        if dates and precip:
            # Create daily dataframe
            df_daily = pd.DataFrame({
                'date': dates,
                'precipitation': precip
            })
            df_daily['date'] = pd.to_datetime(df_daily['date'])
            df_daily = df_daily.set_index('date')
            
            # Resample to monthly sum; keep empty months as NaN instead of 0
            df_monthly = df_daily.resample('ME').sum(min_count=1)
            
            # Add year and month columns
            df_monthly['year'] = df_monthly.index.year
            df_monthly['month'] = df_monthly.index.month
            
            # Pivot to get year as index and month as columns
            pivot_df = df_monthly.pivot(index='year', columns='month', values='precipitation')
            
            # Store it
            dfs_monthly[clave] = pivot_df
            
            print(f"✓ {clave}: {pivot_df.shape[0]} years × 12 months")
        else:
            print(f"❌ {clave}: Could not parse precipitation data")
    
    except Exception as e:
        print(f"❌ {clave}: Error - {str(e)[:50]}")

print(f"\n{'='*60}")
print(f"Successfully created {len(dfs_monthly)} monthly DataFrames")
print(f"{'='*60}")

Processing 16 stations...

✓ 13042: 86 years × 12 months
✓ 30041: 77 years × 12 months
✓ 13087: 72 years × 12 months
✓ 13093: 76 years × 12 months
✓ 13077: 75 years × 12 months
✓ 30180: 66 years × 12 months
✓ 30098: 67 years × 12 months
✓ 13011: 58 years × 12 months
✓ 13015: 84 years × 12 months
✓ 30016: 61 years × 12 months
✓ 30067: 66 years × 12 months
✓ 13139: 45 years × 12 months
✓ 13135: 46 years × 12 months
✓ 30359: 44 years × 12 months
✓ 13137: 35 years × 12 months
✓ 30382: 37 years × 12 months

Successfully created 16 monthly DataFrames


In [8]:
# Filter out years where all months are either NaN or zero
print("Filtering out years where all months are either NaN or zero...\n")

for clave in list(dfs_monthly.keys()):
    df = dfs_monthly[clave].copy()

    # Only inspect month columns (1-12) and coerce anything unexpected to NaN.
    month_cols = [col for col in df.columns if isinstance(col, (int, np.integer)) and 1 <= int(col) <= 12]
    month_data = df[month_cols].apply(pd.to_numeric, errors="coerce")

    num_rows_before = len(df)
    all_missing_or_zero = month_data.isna().all(axis=1) | month_data.eq(0).all(axis=1)
    mixed_missing_and_zero_only = month_data.fillna(0).eq(0).all(axis=1)
    mask = ~mixed_missing_and_zero_only
    df_filtered = df.loc[mask].copy()

    num_rows_after = len(df_filtered)
    dropped = num_rows_before - num_rows_after

    if dropped > 0:
        dropped_years = df.index[~mask].tolist()
        print(f"{clave}: Dropped {dropped} year(s) with only NaNs and zeroes: {dropped_years}")

    dfs_monthly[clave] = df_filtered

print(f"\nFiltering complete.")

Filtering out years where all months are either NaN or zero...

30041: Dropped 3 year(s) with only NaNs and zeroes: [2014, 2015, 2020]
13093: Dropped 6 year(s) with only NaNs and zeroes: [2021, 2022, 2023, 2024, 2025, 2026]
13077: Dropped 7 year(s) with only NaNs and zeroes: [1982, 1994, 2010, 2011, 2012, 2013, 2014]
30180: Dropped 4 year(s) with only NaNs and zeroes: [2010, 2014, 2015, 2016]
30098: Dropped 8 year(s) with only NaNs and zeroes: [1986, 1987, 1988, 1989, 2014, 2015, 2016, 2026]
13015: Dropped 32 year(s) with only NaNs and zeroes: [1969, 1971, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006]
30016: Dropped 11 year(s) with only NaNs and zeroes: [1995, 1996, 1997, 1998, 1999, 2000, 2002, 2003, 2013, 2014, 2015]
30067: Dropped 17 year(s) with only NaNs and zeroes: [1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004,

In [9]:
dfs_monthly["13093"]

month,1,2,3,4,5,6,7,8,9,10,11,12
year,,,,,,,,,,,,
1951,NaN,NaN,NaN,5.61,39.81,56.21,36.02,57.00,84.41,11.02,0.00,NaN
1952,12.00,6.01,0.00,35.50,111.61,107.06,27.35,17.52,293.11,4.40,58.31,0.01
1953,0.00,1.80,7.50,12.10,3.52,26.21,60.62,4.91,102.83,NaN,NaN,NaN
1954,0.30,6.60,3.63,30.31,40.02,31.20,91.56,47.32,203.42,193.20,0.02,0.00
1955,1.81,0.00,12.00,4.20,13.52,7.70,117.33,83.05,416.02,62.03,16.63,3.52
...,...,...,...,...,...,...,...,...,...,...,...,...
2016,NaN,NaN,1.52,NaN,NaN,NaN,40.91,111.41,235.52,34.72,NaN,NaN
2017,0.00,0.00,38.50,NaN,NaN,116.70,42.93,75.83,240.81,105.82,0.20,0.01
2018,8.03,4.22,0.00,9.21,0.00,107.00,12.80,36.80,88.13,49.70,26.20,NaN


## 8. Sample Monthly Precipitation DataFrames

In [10]:
# Show sample of each monthly dataframe
for clave in list(dfs_monthly.keys())[:3]:
    print(f"\nStation {clave}:")
    print(f"Shape: {dfs_monthly[clave].shape}")
    print("\nFirst 5 years:")
    print(dfs_monthly[clave].head())
    print("\n" + "="*80)


Station 13042:
Shape: (86, 12)

First 5 years:
month     1      2      3      4      5       6       7       8       9   \
year                                                                       
1941     NaN    NaN    NaN    NaN    NaN     NaN     NaN     NaN     NaN   
1942   43.17  25.50  84.03    NaN    NaN     NaN  311.02  185.00  599.02   
1943   64.00  24.01  28.02  39.03  36.01  141.04   43.02  160.03  266.01   
1944     NaN  21.04  40.10   0.05  93.06  278.05   81.02  318.02  449.02   
1945   15.18  21.03  33.04   5.52  33.03   88.00   65.84  159.05   79.66   

month      10      11     12  
year                          
1941    45.06  186.65  80.00  
1942    39.02  133.01  35.00  
1943   126.09   43.03  36.02  
1944   147.02   46.57  41.08  
1945    78.92   56.32  18.00  


Station 30041:
Shape: (74, 12)

First 5 years:
month     1     2      3      4      5      6      7      8      9      10  \
year                                                                       

## Summary

This notebook successfully:
1. **Downloaded 29 daily precipitation files** from CONAGUA DIARIOS URLs
2. **Identified 16 climatological stations** with data extending to at least 2015 and at least 20 years of records
3. **Created monthly precipitation DataFrames** for each station with:
   - Year as index
   - Months (1-12) as columns
   - Monthly precipitation totals (mm) as values

The selected stations span from 1941 to 2026, providing a comprehensive dataset for hydrological analysis in the EH 26289 basin.

In [11]:
for clave in list(dfs_monthly.keys()):
    df = dfs_monthly[clave].copy()

    # If a media_mensual row exists, keep it aside (we will reuse it)
    existing_media = None
    if "media_mensual" in df.index:
        # record existing monthly means for later use (if present)
        month_cols_tmp = [c for c in df.columns if isinstance(c, (int, np.integer)) or (isinstance(c, str) and c.isdigit())]
        try:
            existing_media = df.loc["media_mensual", month_cols_tmp]
        except Exception:
            existing_media = None
        df = df.drop(index="media_mensual")

    if "anual" in df.columns:
        df = df.drop(columns=["anual"])

    # Ensure month columns are numeric (coerce non-numeric to NaN)
    month_cols = [c for c in df.columns if isinstance(c, (int, np.integer)) or (isinstance(c, str) and c.isdigit())]
    if month_cols:
        df[month_cols] = df[month_cols].apply(pd.to_numeric, errors='coerce')

    # Drop years with at least one rainy-season month equal to NaN or zero (May-Sep)
    rainy_cols = [m for m in [5, 6, 7, 8, 9] if m in df.columns]
    if rainy_cols:
        # True for rows that have any NaN or any 0 in the rainy months
        mask_bad = df[rainy_cols].isna().any(axis=1) | df[rainy_cols].eq(0).any(axis=1)
        df = df.loc[~mask_bad].copy()

    # Compute monthly means BEFORE filling NaNs. If we had an existing media row, reuse that.
    month_means = None
    if month_cols:
        if existing_media is not None:
            # align to month_cols and coerce to numeric
            month_means = existing_media.reindex(month_cols).astype(float)
        else:
            month_means = df[month_cols].mean(axis=0, skipna=True)

        # Fill NaNs in month columns using the pre-computed monthly means
        df[month_cols] = df[month_cols].fillna(month_means)

    # Recompute annual using filled values
    if month_cols:
        df["anual"] = df[month_cols].sum(axis=1)
    else:
        df["anual"] = df.sum(axis=1)

    # Set media_mensual to the pre-computed means (do NOT recompute from filled data)
    if month_cols and month_means is not None:
        df.loc["media_mensual"] = month_means

    dfs_monthly[clave] = df

In [12]:
dfs_monthly["13093"]

month,1,2,3,4,5,6,7,8,9,10,11,12,anual
year,,,,,,,,,,,,,
1951,9.481967,8.047049,9.401639,5.610000,39.81,56.210000,36.020000,57.000000,84.410,11.020000,0.000000,25.004821,342.015477
1952,12.000000,6.010000,0.000000,35.500000,111.61,107.060000,27.350000,17.520000,293.110,4.400000,58.310000,0.010000,672.880000
1953,0.000000,1.800000,7.500000,12.100000,3.52,26.210000,60.620000,4.910000,102.830,44.644754,12.357167,25.004821,301.496742
1954,0.300000,6.600000,3.630000,30.310000,40.02,31.200000,91.560000,47.320000,203.420,193.200000,0.020000,0.000000,647.580000
1955,1.810000,0.000000,12.000000,4.200000,13.52,7.700000,117.330000,83.050000,416.020,62.030000,16.630000,3.520000,737.810000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2011,1.010000,0.000000,0.010000,32.530000,8.50,282.500000,184.030000,65.500000,94.010,17.010000,28.500000,25.004821,738.604821
2012,9.010000,81.020000,14.510000,13.000000,19.01,75.210000,129.640000,196.110000,70.510,2.610000,5.130000,25.004821,640.764821
2014,3.000000,0.000000,10.220000,9.700000,36.12,212.100000,97.800000,40.000000,67.700,56.300000,25.600000,9.000000,567.540000


In [13]:
dfs_monthly.keys()

dict_keys(['13042', '30041', '13087', '13093', '13077', '30180', '30098', '13011', '13015', '30016', '30067', '13139', '13135', '30359', '13137', '30382'])

In [14]:
dfs_monthly["13093"]

month,1,2,3,4,5,6,7,8,9,10,11,12,anual
year,,,,,,,,,,,,,
1951,9.481967,8.047049,9.401639,5.610000,39.81,56.210000,36.020000,57.000000,84.410,11.020000,0.000000,25.004821,342.015477
1952,12.000000,6.010000,0.000000,35.500000,111.61,107.060000,27.350000,17.520000,293.110,4.400000,58.310000,0.010000,672.880000
1953,0.000000,1.800000,7.500000,12.100000,3.52,26.210000,60.620000,4.910000,102.830,44.644754,12.357167,25.004821,301.496742
1954,0.300000,6.600000,3.630000,30.310000,40.02,31.200000,91.560000,47.320000,203.420,193.200000,0.020000,0.000000,647.580000
1955,1.810000,0.000000,12.000000,4.200000,13.52,7.700000,117.330000,83.050000,416.020,62.030000,16.630000,3.520000,737.810000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2011,1.010000,0.000000,0.010000,32.530000,8.50,282.500000,184.030000,65.500000,94.010,17.010000,28.500000,25.004821,738.604821
2012,9.010000,81.020000,14.510000,13.000000,19.01,75.210000,129.640000,196.110000,70.510,2.610000,5.130000,25.004821,640.764821
2014,3.000000,0.000000,10.220000,9.700000,36.12,212.100000,97.800000,40.000000,67.700,56.300000,25.600000,9.000000,567.540000


In [15]:
f = "../data/precipitacion/precipitacion.xlsx"
p = pd.read_excel(f, sheet_name="13093")
p

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Estación,...,Unnamed: 48,Unnamed: 49,Unnamed: 50,Unnamed: 51,Unnamed: 52,Unnamed: 53,m,Precipitación anual (mm) ordenada,Tr (años),Probabilidad de excedencia (mm)
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,106.040909,46.23,12.9225,23.797797,468.771603,NaN,1.0,1752.740000,67.000000,0.014925
1,Cuenta de Precipitación,Etiquetas de columna,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cuenta de Precipitación,...,NaN,NaN,NaN,NaN,NaN,NaN,2.0,813.140000,33.500000,0.029851
2,Etiquetas de fila,13093,13135.0,13139.0,30016.0,30041.0,30067.0,Total general,NaN,Etiquetas de fila,...,9.000000,10.00,11.0000,12.000000,NaN,NaN,3.0,766.180000,22.333333,0.044776
3,1947,NaN,NaN,NaN,NaN,168.0,NaN,168,NaN,1951,...,84.410000,11.02,0.0000,23.797797,339.741489,NaN,4.0,737.810000,16.750000,0.059701
4,1948,NaN,NaN,NaN,NaN,361.0,NaN,361,NaN,1952,...,293.110000,4.40,58.3100,0.010000,672.880000,NaN,5.0,737.397797,13.400000,0.074627
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,2023,NaN,365.0,365.0,NaN,17.0,360.0,1107,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
80,2024,NaN,363.0,274.0,NaN,NaN,365.0,1002,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
81,2025,58,365.0,90.0,NaN,NaN,365.0,878,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
82,2026,86,90.0,NaN,NaN,NaN,90.0,266,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
excedencias = {}
for clave in list(dfs_monthly.keys()):
    # Work on a copy of the annual series and exclude the monthly-mean row if present
    annual_series = dfs_monthly[clave]["anual"].copy()
    if "media_mensual" in dfs_monthly[clave].index:
        annual_series = annual_series.drop(index="media_mensual")

    # Ensure numeric and drop NA rows
    annual_series = pd.to_numeric(annual_series, errors='coerce').dropna()

    # Sort descending (use all years except the mean row)
    excedencia = pd.DataFrame(annual_series.sort_values(ascending=False))
    excedencia.index = np.arange(1, len(excedencia) + 1)
    excedencia.columns = ["Precip(mm)"]
    excedencia["Tr"] = (len(excedencia) + 1) / excedencia.index
    excedencia["Pe"] = 1 / excedencia["Tr"]
    excedencias[clave] = excedencia

In [17]:
excedencias

{'13042':     Precip(mm)         Tr        Pe
 1      4216.45  78.000000  0.012821
 2      3863.92  39.000000  0.025641
 3      2747.23  26.000000  0.038462
 4      2084.73  19.500000  0.051282
 5      2054.03  15.600000  0.064103
 ..         ...        ...       ...
 73      864.00   1.068493  0.935897
 74      847.70   1.054054  0.948718
 75      804.00   1.040000  0.961538
 76      769.10   1.026316  0.974359
 77      653.59   1.012987  0.987179
 
 [77 rows x 3 columns],
 '30041':     Precip(mm)         Tr        Pe
 1       4163.0  67.000000  0.014925
 2       4134.0  33.500000  0.029851
 3       3204.7  22.333333  0.044776
 4       3057.0  16.750000  0.059701
 5       3056.0  13.400000  0.074627
 ..         ...        ...       ...
 62      1249.7   1.080645  0.925373
 63      1240.5   1.063492  0.940299
 64      1237.0   1.046875  0.955224
 65      1160.1   1.030769  0.970149
 66      1117.3   1.015152  0.985075
 
 [66 rows x 3 columns],
 '13087':     Precip(mm)         Tr       

In [18]:
# Find Precip(mm) corresponding to Pe = 0.85 for each station (linear interpolation where needed)
target_pe = 0.85
precip_at_pe85 = {}
details = {}
for clave, df in excedencias.items():
    # Work on a copy and drop NA rows
    edf = df[['Precip(mm)', 'Pe']].dropna().copy()
    edf = edf.sort_values('Pe').reset_index(drop=True)
    pe_vals = edf['Pe'].values
    p_vals = edf['Precip(mm)'].values
    n = len(pe_vals)
    if n == 0:
        precip_at_pe85[clave] = np.nan
        details[clave] = {'status': 'no_data'}
        continue
    # If exact match exists, take it
    mask_eq = np.isclose(pe_vals, target_pe)
    if mask_eq.any():
        precip_at_pe85[clave] = float(p_vals[mask_eq][0])
        details[clave] = {'status': 'exact'}
        continue
    # If target_pe is outside the empirical range, set NaN (no reliable interpolation)
    if target_pe < pe_vals.min() or target_pe > pe_vals.max():
        precip_at_pe85[clave] = np.nan
        details[clave] = {'status': 'out_of_range', 'min_pe': float(pe_vals.min()), 'max_pe': float(pe_vals.max())}
        continue
    # Locate insertion point and linearly interpolate between neighbouring points
    idx = np.searchsorted(pe_vals, target_pe)
    if idx == 0:
        precip_at_pe85[clave] = float(p_vals[0])
        details[clave] = {'status': 'clamped_low'}
    elif idx >= n:
        precip_at_pe85[clave] = float(p_vals[-1])
        details[clave] = {'status': 'clamped_high'}
    else:
        pe_lo, pe_hi = pe_vals[idx-1], pe_vals[idx]
        p_lo, p_hi = p_vals[idx-1], p_vals[idx]
        # Linear interpolation in (Pe, Precip) space
        t = (target_pe - pe_lo) / (pe_hi - pe_lo)
        precip_interp = p_lo + t * (p_hi - p_lo)
        precip_at_pe85[clave] = float(precip_interp)
        details[clave] = {'status': 'interpolated', 'pe_lo': float(pe_lo), 'pe_hi': float(pe_hi), 'p_lo': float(p_lo), 'p_hi': float(p_hi)}

# Create a summary DataFrame
summary = pd.DataFrame([{'CLAVE': k, 'Precip_at_Pe85_mm': v, 'detail': details.get(k)} for k, v in precip_at_pe85.items()])
summary = summary.set_index('CLAVE').sort_index()
print('Precip(mm) at Pe=0.85 for each station:')
print(summary[['Precip_at_Pe85_mm']])

# Keep results available in the notebook namespace
precip_at_pe85_dict = precip_at_pe85
excedencias_pe85_summary = summary

# Optionally, save to CSV in data folder
try:
    summary.to_csv('../data/precipitacion/excedencias_pe85_summary.csv')
    print('Saved summary to ../data/precipitacion/excedencias_pe85_summary.csv')
except Exception as e:
    print('Could not save summary:', e)

Precip(mm) at Pe=0.85 for each station:
       Precip_at_Pe85_mm
CLAVE                   
13011        1047.510082
13015         248.706000
13042         985.124316
13077         297.980000
13087         253.100000
13093         281.356500
13135        1096.489000
13137         938.875000
13139        1083.580000
30016         198.370000
30041        1403.075000
30067        1029.000000
30098         943.113500
30180        1100.156000
30359         540.956000
30382        1783.554000
Saved summary to ../data/precipitacion/excedencias_pe85_summary.csv


In [19]:
f = "../data/precipitacion/areasThiessen_corregido.csv"
area = pd.read_csv(f)
area = area.set_index("CLAVE")

In [20]:
area["w"] = area["A_km2"] / area["A_km2"].sum()
area

,name,folders,descriptio,altitude,alt_mode,time_begin,time_end,time_when,ALTITUD,CUENCA,...,NORMALES_1,NORMALES_2,NORMALES_3,NORMALES_4,ORG_CUENCA,SITUACION,TIPO_EST,AREA_KM2,A_km2,w
CLAVE,,,,,,,,,,,,,,,,,,,,,
13042,13042,Red Nacional de Estaciones Climatológicas Conv...,NaN,1969.0,NaN,NaN,NaN,NaN,1969,RÍO METZTITLÁN 2,...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,GOLFO NORTE,OPERANDO,CLIMATOLÓGICA,224,223.992565,0.148582
13015,13015,Red Nacional de Estaciones Climatológicas Conv...,NaN,1373.0,NaN,NaN,NaN,NaN,1373,RÍO METZTITLÁN 2,...,NaN,NaN,NaN,NaN,GOLFO NORTE,OPERANDO,CLIMATOLÓGICA,12,11.685194,0.007751
30359,30359,Red Nacional de Estaciones Climatológicas Conv...,NaN,2266.0,NaN,NaN,NaN,NaN,2266,RÍO TUXPAN,...,NaN,NaN,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,GOLFO CENTRO,SUSPENDIDA,CLIMATOLÓGICA,15,15.483681,0.010271
30067,30067,Red Nacional de Estaciones Climatológicas Conv...,NaN,2168.0,NaN,NaN,NaN,NaN,2168,RÍO TUXPAN,...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,NaN,NaN,NaN,GOLFO CENTRO,OPERANDO,CLIMATOLÓGICA,112,112.475988,0.074609
13135,13135,Red Nacional de Estaciones Climatológicas Conv...,NaN,168.0,NaN,NaN,NaN,NaN,168,RÍO LOS HULES,...,NaN,NaN,NaN,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,GOLFO NORTE,OPERANDO,CLIMATOLÓGICA,39,38.807255,0.025742
30382,30382,Red Nacional de Estaciones Climatológicas Conv...,NaN,1612.0,NaN,NaN,NaN,NaN,1612,RÍO CALABOZO,...,NaN,NaN,NaN,NaN,GOLFO NORTE,OPERANDO,CLIMATOLÓGICA,581,581.171406,0.385510
30016,30016,Red Nacional de Estaciones Climatológicas Conv...,NaN,2202.0,NaN,NaN,NaN,NaN,2202,RÍO CALABOZO,...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,NaN,NaN,NaN,GOLFO NORTE,OPERANDO,CLIMATOLÓGICA,311,311.383948,0.206551
13139,13139,Red Nacional de Estaciones Climatológicas Conv...,NaN,503.0,NaN,NaN,NaN,NaN,503,RÍO LOS HULES,...,NaN,NaN,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,GOLFO NORTE,OPERANDO,CLIMATOLÓGICA,98,98.247370,0.065171
30041,30041,Red Nacional de Estaciones Climatológicas Conv...,NaN,291.0,NaN,NaN,NaN,NaN,291,RÍO CALABOZO,...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...,GOLFO NORTE,OPERANDO,CLIMATOLÓGICA,81,81.163964,0.053839


In [21]:
# Build resumen using only CLAVEs present in `area`
# Normalize index types
summary.index = summary.index.astype(str)
area.index = area.index.astype(str)

# Find intersection of CLAVEs present in area and summary
common_claves = area.index.intersection(summary.index)
resumen = summary.loc[common_claves, ['Precip_at_Pe85_mm']].copy()

# Identify numeric area column
area_col = None
candidates = ['A_km2', 'AREA_KM2']
for col in candidates:
    if col in area.columns:
        area_col = col
        break
if area_col is None:
    numeric_cols = area.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        area_col = numeric_cols[-1]
if area_col is None:
    raise RuntimeError('Could not find numeric area column in `area`')

# Subset area to common claves and compute w normalized over that subset
area_sub = area.loc[common_claves].copy()
area_sub['w'] = area_sub[area_col] / area_sub[area_col].sum()

# Join weights
resumen = resumen.join(area_sub[['w']], how='left')
resumen.index.name = 'CLAVE'

# Save
out_dir = Path('../data/precipitacion')
out_dir.mkdir(parents=True, exist_ok=True)
resumen.to_csv(out_dir / 'resumen_pe85_w.csv')
print('Saved resumen to', out_dir / 'resumen_pe85_w.csv')
print(resumen)
resumen

Saved resumen to ..\data\precipitacion\resumen_pe85_w.csv
       Precip_at_Pe85_mm         w
CLAVE                             
13042         985.124316  0.148582
13015         248.706000  0.007751
30359         540.956000  0.010271
30067        1029.000000  0.074609
13135        1096.489000  0.025742
30382        1783.554000  0.385510
30016         198.370000  0.206551
13139        1083.580000  0.065171
30041        1403.075000  0.053839
30180        1100.156000  0.021974


,Precip_at_Pe85_mm,w
CLAVE,,
13042,985.124316,0.148582
13015,248.706000,0.007751
30359,540.956000,0.010271
30067,1029.000000,0.074609
13135,1096.489000,0.025742
30382,1783.554000,0.385510
30016,198.370000,0.206551
13139,1083.580000,0.065171
30041,1403.075000,0.053839


In [22]:
resumen["pw"] = resumen["Precip_at_Pe85_mm"] *resumen["w"]

pa = resumen["pw"].sum()

Ac = area["A_km2"].sum() * 1000000

In [23]:
resumen

,Precip_at_Pe85_mm,w,pw
CLAVE,,,
13042,985.124316,0.148582,146.371446
13015,248.706000,0.007751,1.927764
30359,540.956000,0.010271,5.556072
30067,1029.000000,0.074609,76.772718
13135,1096.489000,0.025742,28.225974
30382,1783.554000,0.385510,687.578398
30016,198.370000,0.206551,40.973582
13139,1083.580000,0.065171,70.617711
30041,1403.075000,0.053839,75.539805


In [32]:
k = 0.251496377750712
ce = k * (pa - 250) / 2000 + (k - 0.15) / 1.5 
cp = pa / 1000 * Ac * ce
cp

np.float64(317320495.9349507)

In [33]:
ce, Ac, pa

(np.float64(0.18181070075003752),
 np.float64(1507538033.7310998),
 np.float64(1157.7383136656836))

In [35]:
Q = cp / (24 * 3600 * 365)
Q

np.float64(10.062166918282303)

In [34]:
2.19+2.25+8.05+5.2+2.48

20.17

In [31]:
Va = ce * Ac * pa / 1000
Va #vol anual escurrido

np.float64(317320495.9349507)